# Phase 4 — Filtrage, correction, livrable

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.config import CFG


## 0. Modèle de langue gcf (KenLM ou repli caractère)

In [ ]:
from src.build_lm import build_parser as lm_parser, main as build_lm
build_lm(lm_parser().parse_args(['--order', '5']))


## 1-3. Scoring, filtrage, correction, écriture des livrables

`--calibrate` fixe le seuil de perplexité empiriquement sur du gcf propre, plutôt qu'à la valeur codée en dur (100) dont le sens dépend du modèle utilisé.

In [ ]:
from src.phase4_filter_decode import build_parser, run
stats = run(build_parser().parse_args(['--calibrate', '--audit-sample', '200']))
stats


## Distribution des confiances

In [ ]:
import pandas as pd
from src.io_utils import read_jsonl
df = pd.DataFrame(read_jsonl(CFG.dataset_corrected))
df[df.filtre_passe][['confiance','wer_estime','taux_edition','perplexite_corrigee']].describe()


## Validation humaine — indispensable

Le WER du livrable est **estimé**, pas mesuré : il n'existe pas de vérité terrain sur le corpus Whisper. Remplir `artifacts/audit_humain.csv` puis comparer le WER réel au WER estimé est le seul moyen de savoir si le livrable tient sa promesse.

In [ ]:
import csv, pathlib
from src.metrics import wer
p = pathlib.Path(CFG.dataset_high_conf).with_name('audit_humain.csv')
rows = [r for r in csv.DictReader(p.open(encoding='utf-8')) if r['verite_terrain'].strip()]
if rows:
    reel = sum(wer(r['verite_terrain'], r['gcf_corrige']) for r in rows) / len(rows)
    est  = sum(float(r['wer_estime']) for r in rows) / len(rows)
    print(f'WER réel {reel:.4f} vs estimé {est:.4f} sur {len(rows)} segments annotés')
else:
    print("Aucune annotation : le WER annoncé reste une extrapolation non vérifiée.")


## Rapport final

In [ ]:
import argparse
from src.report import main as report
report(argparse.Namespace(out=None))
